<a href="https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [31]:
!pip -q install duckdb
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE HUGGINGFACE, TOKEN '{userdata.get('HF_TOKEN')}')")

## 2) Fields

| Bucket | Fields |
|---|---|
| Feature | impressions_early, ctr_early, avg_position_early, days_since_update, word_count |
| Label | label (late-window clicks per day below 80% of the early rate) |
| Context | client_hash_id (for the grouped split), content_hash_id |
| Excluded | late_clicks_per_day (future window, would leak); anything dated after 20 March; fact_content_query_90d (query text may be private); the June 2026 sealed month |

In [32]:
!pip -q install huggingface_hub
from huggingface_hub import HfApi
from google.colab import userdata

api = HfApi(token=userdata.get("HF_TOKEN"))
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files[:60]:
    print(f)
print("total files:", len(files))

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [33]:
from huggingface_hub import hf_hub_download
from google.colab import userdata

try:
    p = hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        repo_type="dataset",
        filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
        token=userdata.get("HF_TOKEN"),
    )
    print("OK, downloaded to:", p)
except Exception as e:
    print(type(e).__name__, str(e)[:300])

OK, downloaded to: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [34]:
from huggingface_hub import HfApi
from google.colab import userdata

tok = userdata.get("HF_TOKEN")
print("Token starts with hf_:", tok.startswith("hf_"), "| length:", len(tok))

api = HfApi(token=tok)
try:
    print("Logged in as:", api.whoami()["name"])
except Exception as e:
    print("whoami failed:", type(e).__name__, str(e)[:200])

Token starts with hf_: True | length: 37
Logged in as: janaadarwish


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [35]:
import duckdb
from huggingface_hub import hf_hub_download
from google.colab import userdata

TOKEN = userdata.get("HF_TOKEN")
REPO = "FlyRank/internship-warehouse"

try:
    FACT = hf_hub_download(REPO, "fact_content_daily_performance/month=2026-03/data_0.parquet",
                           repo_type="dataset", token=TOKEN)
    DIM_CONTENT = hf_hub_download(REPO, "dim_content.parquet", repo_type="dataset", token=TOKEN)
    DIM_CLIENTS = hf_hub_download(REPO, "dim_clients.parquet", repo_type="dataset", token=TOKEN)
    print("Downloads OK")
except Exception as e:
    print(type(e).__name__, str(e)[:300])
    raise

con = duckdb.connect()
for name, path in [("FACT", FACT), ("dim_content", DIM_CONTENT), ("dim_clients", DIM_CLIENTS)]:
    print("=====", name)
    print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()[["column_name", "column_type"]].to_string())

Downloads OK
===== FACT
                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt     

## 1) The contract

1. **One row means:** one page (`content_hash_id`) of one client (`client_hash_id`) on one day (`report_date`) in the daily performance table. For modeling, I roll this up to one row per page.
2. **Tables used:** `fact_content_daily_performance` (month=2026-03) and `dim_content` (page details such as updated date and word count).
3. **Time window:** features come from 1 to 20 March 2026. The label looks at 21 to 31 March 2026. The final month (June 2026) stays sealed.
4. **What I rank:** Pages whose daily clicks in 21 to 31 March fell below 80% of their 1 to 20 March rate (label = 1), as a proxy for ‘worth reviewing’. It is a proxy, not proof a page needs a refresh.
5. **One thing I deliberately exclude:** pages whose `content_updated_date` is after 20 March, because their update date would be unknown at the decision moment.

In [36]:
con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM read_parquet('{FACT}')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


In [37]:
con.sql(f"""
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT content_hash_id) AS n_pages,
       MIN(report_date) AS first_day,
       MAX(report_date) AS last_day
FROM read_parquet('{FACT}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_pages,first_day,last_day
0,9841378,331437,2026-03-01,2026-03-31


In [38]:
con.sql(f"""
SELECT COUNT(*) AS all_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
FROM read_parquet('{FACT}')
""").df()

,all_rows,gsc_available_rows
0,9841378,3611061


In [39]:
df = con.sql(f"""
WITH early AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS impressions_early,
         SUM(gsc_clicks) AS clicks_early,
         AVG(gsc_avg_position) AS avg_position_early
  FROM read_parquet('{FACT}')
  WHERE gsc_data_available IS TRUE AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-20'
  GROUP BY 1, 2
),
late AS (
  SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_late
  FROM read_parquet('{FACT}')
  WHERE gsc_data_available IS TRUE AND report_date BETWEEN DATE '2026-03-21' AND DATE '2026-03-31'
  GROUP BY 1, 2
)
SELECT e.client_hash_id, e.content_hash_id,
       e.impressions_early,
       e.clicks_early * 1.0 / NULLIF(e.impressions_early, 0) AS ctr_early,
       e.avg_position_early,
       DATE '2026-03-20' - c.content_updated_date AS days_since_update,
       c.word_count,
       COALESCE(l.clicks_late, 0) / 11.0 AS late_clicks_per_day,       -- future window, used for the label only
       (COALESCE(l.clicks_late, 0) / 11.0) < 0.8 * (e.clicks_early / 20.0) AS label
FROM early e
JOIN read_parquet('{DIM_CONTENT}') c USING (client_hash_id, content_hash_id)
LEFT JOIN late l USING (client_hash_id, content_hash_id)
WHERE e.clicks_early >= 5
  AND c.content_updated_date <= DATE '2026-03-20'
""").df()

df["label"] = df["label"].astype(int)
print(df.shape, "| share label=1:", round(df["label"].mean(), 3))
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(2526, 9) | share label=1: 0.59


,client_hash_id,content_hash_id,impressions_early,ctr_early,avg_position_early,days_since_update,word_count,late_clicks_per_day,label
0,client_23a62021009f63c4,content_622dc07ba2bf2fc4,2085.0,0.002878,15.323217,23,6956,0.181818,1
1,client_23a62021009f63c4,content_86045ee7c4e663a3,487.0,0.012320,6.396862,23,5211,0.090909,1
2,client_23a62021009f63c4,content_8f7c81769e5ef3f8,11496.0,0.001131,40.907858,23,4626,0.818182,0
3,client_23a62021009f63c4,content_4fe94bdfd75c38f9,8411.0,0.002140,37.221918,23,6645,0.727273,0
4,client_23a62021009f63c4,content_f7db7c9b2e37e149,15204.0,0.000526,39.554873,23,6609,0.181818,1


### Features and when each is knowable

- **impressions_early:** knowable at the decision moment because it sums impressions up to 20 March, before the label window starts.
- **ctr_early:** knowable because it is early clicks divided by early impressions, both measured by 20 March.
- **avg_position_early:** knowable because it averages positions reported up to 20 March.
- **days_since_update:** knowable because it is 20 March minus the page's last update date, and I excluded pages updated after 20 March.
- **word_count:** knowable because page length is known when the page is published.
- **search_volume:** knowable because it describes the target keyword and comes from the content table, not from the label window.

In [40]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

FEATS = ["impressions_early", "ctr_early", "avg_position_early", "days_since_update", "word_count"]
d = df.dropna(subset=FEATS + ["label"]).reset_index(drop=True)

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
tr, te = next(gss.split(d, groups=d["client_hash_id"]))

def score(cols):
    m = DecisionTreeClassifier(max_depth=4, random_state=0).fit(d.loc[tr, cols], d.loc[tr, "label"])
    return round(roc_auc_score(d.loc[te, "label"], m.predict_proba(d.loc[te, cols])[:, 1]), 3)

honest = score(FEATS)
print("Honest score (AUC):", honest)

Honest score (AUC): 0.526


In [41]:
leaky = score(FEATS + ["late_clicks_per_day"])   # derived from the same future window as the label
print("Leaky score (AUC):", leaky)

Leaky score (AUC): 0.888


In [42]:
print("Back to the honest score:", honest, "| the leaky score was:", leaky)

Back to the honest score: 0.526 | the leaky score was: 0.888


In [43]:
print("rows used:", len(d), "of", len(df))
print(df[FEATS].isna().mean().round(3))

rows used: 1345 of 2526
impressions_early     0.000
ctr_early             0.000
avg_position_early    0.000
days_since_update     0.000
word_count            0.468
dtype: float64


In [44]:
print(df.merge(
    con.sql(f"SELECT client_hash_id, content_hash_id, search_volume FROM read_parquet('{DIM_CONTENT}')").df(),
    on=["client_hash_id", "content_hash_id"], how="left"
)["search_volume"].isna().mean().round(3))

0.016


In [45]:
FEATS = ["impressions_early", "ctr_early", "avg_position_early", "days_since_update", "search_volume"]

## The trap

I added `late_clicks_per_day`, which is computed from the same late window as the label. The score jumped from 0.526 to 0.888 because the model was handed the answer. I removed it and kept the honest score of 0.526.

## 4) One named limitation of my slice

Only 2,526 of 331,437 pages pass the filters (GSC data available, at least 5 early clicks, not updated after 20 March), so results describe a small, high-traffic subset, not all pages. Also, `dim_content` is a snapshot taken at release time, not at 20 March, so some page details may have changed since then. I reduced that risk by excluding pages updated after 20 March, but I can't rule it out. The label is also a proxy.

## 5) Self-check

- [✔] Five plain-words contract answers
- [✔] Exactly three verification queries, with the availability check using IS TRUE
- [✔] Five features, each with a "knowable when" line
- [✔] Deliberate leak shown and removed
- [✔] One limitation named